# AMOS22 Adrenal Gland Segmentation Experiments

Use this notebook to inspect native AMOS22 NIfTI volumes, visualize adrenal annotations, test the existing 2.5D U-Net code, run small experiments, and evaluate predictions. AMOS22 label IDs are configurable below; the local `dataset.json` currently defines right adrenal gland as **11** and left adrenal gland as **12**.

> The default settings are deliberately small for exploratory runs. They are not intended as final benchmark settings or clinical validation.

## 1. Configure Paths, Labels, and Experiment Settings

Edit this cell first when moving the notebook to another machine. The defaults locate the project and the adjacent AMOS22 dataset automatically. The label IDs are checked against `dataset.json` before use.

In [ ]:
import os
from pathlib import Path

# Files that together identify the repository root and nothing else.
PROJECT_SENTINELS = ("src/data/preprocessing.py", "src/models/unet25d.py", "configs/default.yaml")


def find_project_root(start: Path) -> Path:
    """Walk up from `start` until a directory holds every sentinel file.

    Set ADRENAL_SEGMENTATION_ROOT to skip the search and name the repository
    directly. The extra `start / "adrenal_segmentation"` candidate covers
    launching Jupyter from the folder *containing* the clone.
    """
    override = os.environ.get("ADRENAL_SEGMENTATION_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents, start / "adrenal_segmentation"]

    for candidate in candidates:
        if all((candidate / sentinel).is_file() for sentinel in PROJECT_SENTINELS):
            return candidate

    raise FileNotFoundError(
        f"No directory containing {PROJECT_SENTINELS} was found from {start}. "
        "Set ADRENAL_SEGMENTATION_ROOT to the cloned repository path."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT.parent / "data" / "amos22"
IMAGE_DIRS = {"train": DATA_ROOT / "imagesTr", "validation": DATA_ROOT / "imagesVa"}
LABEL_DIRS = {"train": DATA_ROOT / "labelsTr", "validation": DATA_ROOT / "labelsVa"}
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "adrenal_unet25d.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_experiments"

# Verified against the local AMOS22 dataset.json; keep configurable for other dataset versions.
RIGHT_ADRENAL_LABEL = 11
LEFT_ADRENAL_LABEL = 12
COMBINE_GLANDS = True

SEED = 42
DEVICE_PREFERENCE = "auto"  # "auto", "cuda", or "cpu"
NUM_TEST_RUNS = 2
TRAIN_PATIENTS = 3
VAL_PATIENTS = 1
MAX_POSITIVE_SLICES_PER_PATIENT = 24
NEGATIVE_TO_POSITIVE_RATIO = 0.5
SLICE_WINDOW = 5
IMAGE_SIZE = (192, 192)
HU_WINDOW = (-135.0, 215.0)
BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATES = [3e-4, 1e-4]
ENCODER = "resnet18"  # Use "inceptionv4" for the configured project architecture.
ENCODER_WEIGHTS = None  # Avoids downloading pretrained weights during a smoke run.
NUM_WORKERS = 0
PREDICTION_THRESHOLD = 0.5
MIN_COMPONENT_VOXELS = 30
RUN_TRAINING = True
RUN_FULL_VOLUME_INFERENCE = True
SAVE_OUTPUTS = False

print(f"Kernel working directory: {Path.cwd().resolve()}")
print(f"Project: {PROJECT_ROOT}")
print(f"AMOS22:  {DATA_ROOT}")
print(f"Output:  {OUTPUT_DIR}")

## 2. Import Libraries and Project Code

The notebook uses the existing project model, loss, preprocessing, evaluation, and post-processing functions. If an import is missing, run this after the configuration cell, then restart the active kernel:

```python
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-r", str(PROJECT_ROOT / "requirements.txt"),
    "matplotlib", "jupyter",
])
```

In [ ]:
import gc
import json
import random
import sys
import time
import warnings

# Index 0 so this project's `src` package wins over any same-named installed
# package, without duplicating the entry on re-run.
sys.path = [str(PROJECT_ROOT)] + [entry for entry in sys.path if entry != str(PROJECT_ROOT)]

# A partially-imported `src` left by an earlier failed run would shadow the
# real package for the rest of the kernel's life.
for module_name in [name for name in sys.modules if name == "src" or name.startswith("src.")]:
    del sys.modules[module_name]

try:
    import matplotlib.pyplot as plt
    import nibabel as nib
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
except ImportError as exc:
    raise ImportError(
        f"Missing notebook dependency: {exc.name}. Run the installation command in the preceding cell, "
        "restart the kernel, and run from the top."
    ) from exc

from src.data.preprocessing import PreprocessConfig, apply_hu_window, preprocess_volume
from src.evaluation.metrics import (
    dice_score,
    hausdorff_distance_95,
    iou_score,
    normalized_surface_dice,
    volume_error,
)
from src.models.losses import DiceFocalLoss
from src.models.unet25d import build_unet25d
from src.postprocessing.connected_components import remove_small_components

DEVICE = torch.device(
    "cuda" if DEVICE_PREFERENCE == "auto" and torch.cuda.is_available() else
    "cpu" if DEVICE_PREFERENCE == "auto" else DEVICE_PREFERENCE
)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import src

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")
print(f"Project src: {Path(src.__file__).resolve()}")

## 3. Discover and Match AMOS22 Cases

Cases are matched by filename and kept in the official training or validation split. AMOS IDs below 500 are CT according to the dataset README.

In [ ]:
dataset_json_path = DATA_ROOT / "dataset.json"
if not dataset_json_path.is_file():
    raise FileNotFoundError(f"Missing AMOS metadata: {dataset_json_path}")

with dataset_json_path.open("r", encoding="utf-8") as file:
    dataset_metadata = json.load(file)

label_names = {int(label_id): name for label_id, name in dataset_metadata["labels"].items()}
assert label_names.get(RIGHT_ADRENAL_LABEL) == "right adrenal gland", (
    f"Configured right label {RIGHT_ADRENAL_LABEL!r} does not match dataset.json: {label_names}"
)
assert label_names.get(LEFT_ADRENAL_LABEL) == "left adrenal gland", (
    f"Configured left label {LEFT_ADRENAL_LABEL!r} does not match dataset.json: {label_names}"
)


def case_number(path: Path) -> int:
    return int(path.name.removeprefix("amos_").removesuffix(".nii.gz"))


def discover_pairs(split: str) -> list[dict]:
    image_dir, label_dir = IMAGE_DIRS[split], LABEL_DIRS[split]
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f"Missing {split} directories: {image_dir} or {label_dir}")
    image_paths = {path.name: path for path in image_dir.glob("*.nii.gz")}
    label_paths = {path.name: path for path in label_dir.glob("*.nii.gz")}
    missing_labels = sorted(set(image_paths) - set(label_paths))
    if missing_labels:
        raise FileNotFoundError(f"{len(missing_labels)} images have no label, including {missing_labels[:3]}")
    return [
        {
            "split": split,
            "case_id": filename.removesuffix(".nii.gz"),
            "image_path": image_paths[filename],
            "label_path": label_paths[filename],
        }
        for filename in sorted(image_paths)
        if case_number(image_paths[filename]) < 500
    ]


train_records = discover_pairs("train")
validation_records = discover_pairs("validation")
all_records = train_records + validation_records
cases_df = pd.DataFrame(all_records)

print(cases_df.groupby("split").size().rename("paired_ct_cases"))
display(cases_df.head())

## 4. Load and Inspect NIfTI Volumes

Nibabel loads arrays in `(X, Y, Z)` order. The helper below also creates `(Z, H, W)` arrays for the project’s axial-slice convention while retaining affine, spacing, and orientation metadata.

In [ ]:
def load_amos_case(record: dict) -> dict:
    image_nifti = nib.load(str(record["image_path"]))
    label_nifti = nib.load(str(record["label_path"]))

    if image_nifti.shape != label_nifti.shape:
        raise ValueError(f"Shape mismatch: image {image_nifti.shape}, label {label_nifti.shape}")
    if not np.allclose(image_nifti.affine, label_nifti.affine, atol=1e-4):
        raise ValueError("Image and label affines do not match")

    image_xyz = np.asarray(image_nifti.dataobj, dtype=np.float32)
    label_xyz = np.asarray(label_nifti.dataobj, dtype=np.int16)
    return {
        "case_id": record["case_id"],
        "image": np.transpose(image_xyz, (2, 0, 1)),
        "label": np.transpose(label_xyz, (2, 0, 1)),
        "image_nifti": image_nifti,
        "label_nifti": label_nifti,
        "spacing_xyz": tuple(float(value) for value in image_nifti.header.get_zooms()[:3]),
        "spacing_zyx": tuple(float(value) for value in image_nifti.header.get_zooms()[:3][::-1]),
        "orientation": nib.aff2axcodes(image_nifti.affine),
    }


example_case = load_amos_case(train_records[0])
print(f"Case:             {example_case['case_id']}")
print(f"Native XYZ shape: {example_case['image_nifti'].shape}")
print(f"Project ZHW:      {example_case['image'].shape}")
print(f"Spacing XYZ (mm): {example_case['spacing_xyz']}")
print(f"Orientation:      {example_case['orientation']}")
print(f"Image dtype/HU:   {example_case['image'].dtype}, "
      f"[{example_case['image'].min():.1f}, {example_case['image'].max():.1f}]")
print(f"Label dtype:      {example_case['label'].dtype}")
print(f"Unique labels:    {np.unique(example_case['label']).tolist()}")
print("Affine:\n", example_case["image_nifti"].affine)

## 5. Extract Adrenal Gland Masks

The binary masks remain separate for measurement and visualization. A combined mask is used by the default one-channel exploratory model.

In [ ]:
def extract_adrenal_masks(label_volume: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    right_mask = label_volume == RIGHT_ADRENAL_LABEL
    left_mask = label_volume == LEFT_ADRENAL_LABEL
    if not right_mask.any():
        warnings.warn(f"Right adrenal label {RIGHT_ADRENAL_LABEL} is absent")
    if not left_mask.any():
        warnings.warn(f"Left adrenal label {LEFT_ADRENAL_LABEL} is absent")
    return right_mask, left_mask, right_mask | left_mask


right_mask, left_mask, combined_mask = extract_adrenal_masks(example_case["label"])
voxel_volume_mm3 = float(np.prod(example_case["spacing_xyz"]))
mask_summary = pd.DataFrame([
    {"structure": "right adrenal", "label": RIGHT_ADRENAL_LABEL, "voxels": int(right_mask.sum())},
    {"structure": "left adrenal", "label": LEFT_ADRENAL_LABEL, "voxels": int(left_mask.sum())},
    {"structure": "combined", "label": f"{RIGHT_ADRENAL_LABEL}|{LEFT_ADRENAL_LABEL}", "voxels": int(combined_mask.sum())},
])
mask_summary["volume_ml"] = mask_summary["voxels"] * voxel_volume_mm3 / 1000.0
mask_summary["positive_axial_slices"] = [
    int(np.any(mask, axis=(1, 2)).sum()) for mask in (right_mask, left_mask, combined_mask)
]
display(mask_summary.round({"volume_ml": 3}))

## 6. Visualize Image Slices and Ground-Truth Masks

Right adrenal is shown in red and left adrenal in cyan. The informative axial, coronal, and sagittal indices are selected from the combined mask’s center of mass.

In [ ]:
def mask_overlay(right: np.ndarray, left: np.ndarray) -> np.ndarray:
    overlay = np.zeros((*right.shape, 4), dtype=np.float32)
    overlay[right] = (1.0, 0.15, 0.10, 0.55)
    overlay[left] = (0.00, 0.85, 0.95, 0.55)
    return overlay


def show_triplanar(case: dict) -> None:
    image = apply_hu_window(case["image"], HU_WINDOW)
    right, left, combined = extract_adrenal_masks(case["label"])
    if combined.any():
        axial, coronal, sagittal = np.round(np.argwhere(combined).mean(axis=0)).astype(int)
    else:
        axial, coronal, sagittal = np.array(combined.shape) // 2

    views = [
        (image[axial], right[axial], left[axial], f"Axial z={axial}"),
        (image[:, coronal, :], right[:, coronal, :], left[:, coronal, :], f"Coronal y={coronal}"),
        (image[:, :, sagittal], right[:, :, sagittal], left[:, :, sagittal], f"Sagittal x={sagittal}"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for column, (image_view, right_view, left_view, title) in enumerate(views):
        axes[0, column].imshow(image_view, cmap="gray", origin="lower")
        axes[0, column].set_title(title)
        axes[1, column].imshow(image_view, cmap="gray", origin="lower")
        axes[1, column].imshow(mask_overlay(right_view, left_view), origin="lower")
        axes[1, column].set_title("Right (red) / Left (cyan)")
    for axis in axes.ravel():
        axis.axis("off")
    plt.show()


show_triplanar(example_case)

## 7. Apply Model Preprocessing

The current project `PatientVolume` path applies HU clipping and volume-level zero-mean/unit-variance normalization, then the 2.5D model consumes neighboring axial slices as channels. This notebook preserves that behavior and resizes each window to control memory. Masks always use nearest-neighbor interpolation. The project exposes a separate physical-spacing resampler, but it is not called automatically by `PatientVolume`; enable and validate resampling before final benchmark runs.

In [ ]:
PREPROCESS_CONFIG = PreprocessConfig(hu_window=HU_WINDOW)


def preprocess_image_volume(image_zhw: np.ndarray) -> np.ndarray:
    """HU window -> volume-level z-score, exactly as the project pipeline does."""
    return preprocess_volume(image_zhw, PREPROCESS_CONFIG)


def resize_window(window: np.ndarray, image_size=IMAGE_SIZE) -> torch.Tensor:
    tensor = torch.from_numpy(np.ascontiguousarray(window)).float().unsqueeze(0)
    return F.interpolate(tensor, size=image_size, mode="bilinear", align_corners=False).squeeze(0)


def resize_mask(mask: np.ndarray, image_size=IMAGE_SIZE) -> torch.Tensor:
    tensor = torch.from_numpy(np.ascontiguousarray(mask)).float()[None, None]
    return F.interpolate(tensor, size=image_size, mode="nearest").squeeze(0)


def make_slice_window(image: np.ndarray, center: int, width: int = SLICE_WINDOW) -> np.ndarray:
    """Neighbouring slices as channels, edge-clamped at the volume boundary to
    match `PatientVolume.get_window`, so notebook experiments and pipeline
    runs cannot silently diverge."""
    if width < 1 or width % 2 == 0:
        raise ValueError("SLICE_WINDOW must be a positive odd integer")
    indices = np.clip(center + np.arange(-(width // 2), width // 2 + 1), 0, image.shape[0] - 1)
    return image[indices]


positive_slices = np.flatnonzero(np.any(combined_mask, axis=(1, 2)))
example_center = int(positive_slices[len(positive_slices) // 2])
example_preprocessed = preprocess_image_volume(example_case["image"])
example_window = make_slice_window(example_preprocessed, example_center)
example_tensor = resize_window(example_window)
example_target = resize_mask(combined_mask[example_center])

fig, axes = plt.subplots(1, SLICE_WINDOW + 1, figsize=(16, 3), constrained_layout=True)
for index in range(SLICE_WINDOW):
    axes[index].imshow(example_tensor[index], cmap="gray")
    axes[index].set_title(f"Channel {index + 1}")
axes[-1].imshow(example_target[0], cmap="gray")
axes[-1].set_title("Target")
for axis in axes:
    axis.axis("off")
plt.show()
print(f"Input tensor: {tuple(example_tensor.shape)}; target tensor: {tuple(example_target.shape)}")

In [ ]:
class AMOSAdrenalSliceDataset(Dataset):
    """Patient-level AMOS sampler with capped positives and sampled negatives."""

    def __init__(self, records, augment=False, seed=SEED):
        if SLICE_WINDOW < 1 or SLICE_WINDOW % 2 == 0:
            raise ValueError("SLICE_WINDOW must be a positive odd integer")
        self.records = list(records)
        self.augment = augment
        self.rng = random.Random(seed)
        self.samples = []
        self._cache_index = None
        self._cache_image = None
        self._cache_label = None

        for record_index, record in enumerate(self.records):
            labels_xyz = np.asarray(nib.load(str(record["label_path"])).dataobj, dtype=np.int16)
            labels = np.transpose(labels_xyz, (2, 0, 1))
            adrenal = (labels == RIGHT_ADRENAL_LABEL) | (labels == LEFT_ADRENAL_LABEL)
            all_positive_slices = np.flatnonzero(np.any(adrenal, axis=(1, 2))).tolist()
            positives = list(all_positive_slices)
            if not positives:
                warnings.warn(f"Skipping {record['case_id']}: no adrenal labels")
                continue

            if len(positives) > MAX_POSITIVE_SLICES_PER_PATIENT:
                positions = np.linspace(0, len(positives) - 1, MAX_POSITIVE_SLICES_PER_PATIENT).round().astype(int)
                positives = [positives[position] for position in positions]

            all_positive = set(all_positive_slices)
            margin = 12
            nearby = [
                index for index in range(max(0, min(all_positive) - margin), min(labels.shape[0], max(all_positive) + margin + 1))
                if index not in all_positive
            ]
            distant = [index for index in range(labels.shape[0]) if index not in all_positive and index not in nearby]
            negative_count = int(round(len(positives) * NEGATIVE_TO_POSITIVE_RATIO))
            self.rng.shuffle(nearby)
            self.rng.shuffle(distant)
            negatives = (nearby + distant)[:negative_count]

            self.samples.extend((record_index, center, True) for center in positives)
            self.samples.extend((record_index, center, False) for center in negatives)

        if not self.samples:
            raise ValueError("No adrenal slice samples were discovered")
        self.rng.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def _load_record(self, record_index):
        if self._cache_index != record_index:
            case = load_amos_case(self.records[record_index])
            self._cache_index = record_index
            self._cache_image = preprocess_image_volume(case["image"])
            self._cache_label = case["label"]
        return self._cache_image, self._cache_label

    def __getitem__(self, index):
        record_index, center, is_positive = self.samples[index]
        image, label = self._load_record(record_index)
        window = resize_window(make_slice_window(image, center))
        target = (label[center] == RIGHT_ADRENAL_LABEL) | (label[center] == LEFT_ADRENAL_LABEL)
        target = resize_mask(target)

        if self.augment:
            if self.rng.random() < 0.5:
                window, target = torch.flip(window, (-1,)), torch.flip(target, (-1,))
            if self.rng.random() < 0.5:
                window, target = torch.flip(window, (-2,)), torch.flip(target, (-2,))

        return {
            "slices": window,
            "mask": target,
            "case_id": self.records[record_index]["case_id"],
            "center_index": center,
            "is_positive": is_positive,
        }


selected_train_records = train_records[:TRAIN_PATIENTS]
selected_validation_records = validation_records[:VAL_PATIENTS]
train_dataset = AMOSAdrenalSliceDataset(selected_train_records, augment=True, seed=SEED)
validation_dataset = AMOSAdrenalSliceDataset(selected_validation_records, augment=False, seed=SEED + 1)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

sample_batch = next(iter(train_loader))
print("Train cases:", [record["case_id"] for record in selected_train_records])
print("Validation cases:", [record["case_id"] for record in selected_validation_records])
print(f"Samples: train={len(train_dataset)}, validation={len(validation_dataset)}")
print(f"Batch: slices={tuple(sample_batch['slices'].shape)}, masks={tuple(sample_batch['mask'].shape)}")

## 8. Load the Segmentation Model

A checkpoint is loaded when `CHECKPOINT_PATH` exists. Otherwise, the model starts with fresh weights for the small training runs below. The default ResNet18 encoder is faster for notebook experiments; switch `ENCODER` to `inceptionv4` to match the project configuration.

In [ ]:
def create_model() -> torch.nn.Module:
    return build_unet25d(
        encoder=ENCODER,
        in_channels=SLICE_WINDOW,
        num_classes=1,
        encoder_weights=ENCODER_WEIGHTS,
    ).to(DEVICE)


model = create_model()
if CHECKPOINT_PATH.is_file():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    load_result = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded checkpoint: {CHECKPOINT_PATH}")
    print("Missing keys:", load_result.missing_keys)
    print("Unexpected keys:", load_result.unexpected_keys)
else:
    print(f"No checkpoint at {CHECKPOINT_PATH}; using fresh weights.")
model.eval()
print(f"Parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

### Train a Few Exploratory Runs

Each run starts from fresh weights. With the default three training patients, one validation patient, and three epochs, these are pipeline smoke tests rather than meaningful final models.

In [ ]:
def batch_dice(logits: torch.Tensor, targets: torch.Tensor) -> float:
    predictions = (torch.sigmoid(logits) >= PREDICTION_THRESHOLD).cpu().numpy()
    references = targets.cpu().numpy()
    return float(np.mean([
        dice_score(prediction[0], reference[0])
        for prediction, reference in zip(predictions, references)
    ]))


def run_epoch(model, loader, criterion, optimizer=None) -> dict:
    is_training = optimizer is not None
    model.train(is_training)
    losses, dice_values = [], []
    context = torch.enable_grad() if is_training else torch.inference_mode()
    with context:
        for batch in loader:
            inputs = batch["slices"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = criterion(logits, targets)
            if is_training:
                loss.backward()
                optimizer.step()
            losses.append(float(loss.item()))
            dice_values.append(batch_dice(logits.detach(), targets))
    return {"loss": float(np.mean(losses)), "dice": float(np.mean(dice_values))}


def train_run(run_name: str, learning_rate: float, epochs: int = EPOCHS):
    run_model = create_model()
    criterion = DiceFocalLoss()
    optimizer = torch.optim.AdamW(run_model.parameters(), lr=learning_rate)
    rows = []
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        train_metrics = run_epoch(run_model, train_loader, criterion, optimizer)
        validation_metrics = run_epoch(run_model, validation_loader, criterion)
        row = {
            "run": run_name,
            "epoch": epoch,
            "learning_rate": learning_rate,
            "train_loss": train_metrics["loss"],
            "val_loss": validation_metrics["loss"],
            "train_dice": train_metrics["dice"],
            "val_dice": validation_metrics["dice"],
            "seconds": time.perf_counter() - started,
        }
        rows.append(row)
        print(
            f"{run_name} | {epoch:02d}/{epochs} | "
            f"loss {row['train_loss']:.4f}/{row['val_loss']:.4f} | "
            f"dice {row['train_dice']:.4f}/{row['val_dice']:.4f} | {row['seconds']:.1f}s"
        )
    return run_model, pd.DataFrame(rows)


models, histories = {}, {}
if RUN_TRAINING:
    for run_index, learning_rate in enumerate(LEARNING_RATES[:NUM_TEST_RUNS], start=1):
        run_name = f"run_{run_index}_lr_{learning_rate:g}"
        try:
            models[run_name], histories[run_name] = train_run(run_name, learning_rate)
        except torch.cuda.OutOfMemoryError:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            raise RuntimeError("CUDA ran out of memory. Reduce IMAGE_SIZE or BATCH_SIZE and rerun.")
    history_df = pd.concat(histories.values(), ignore_index=True)
    best_run = history_df.loc[history_df.groupby("run")["val_loss"].idxmin()].sort_values("val_loss").iloc[0]["run"]
    model = models[best_run]
    print(f"Selected model: {best_run}")
    display(history_df)
else:
    history_df = pd.DataFrame()
    best_run = "loaded_checkpoint" if CHECKPOINT_PATH.is_file() else "fresh_untrained"

model.eval()

In [ ]:
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for run_name, run_history in history_df.groupby("run"):
        axes[0].plot(run_history["epoch"], run_history["train_loss"], marker="o", label=f"{run_name} train")
        axes[0].plot(run_history["epoch"], run_history["val_loss"], marker="s", linestyle="--", label=f"{run_name} val")
        axes[1].plot(run_history["epoch"], run_history["train_dice"], marker="o", label=f"{run_name} train")
        axes[1].plot(run_history["epoch"], run_history["val_dice"], marker="s", linestyle="--", label=f"{run_name} val")
    axes[0].set(title="Loss", xlabel="Epoch", ylabel="Dice + focal loss")
    axes[1].set(title="Thresholded slice Dice", xlabel="Epoch", ylabel="Dice")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)
    plt.show()

## 9. Run Inference on a Single Case

This first pass records tensor shapes, runtime, and peak CUDA memory. It uses the selected trained model, loaded checkpoint, or fresh model according to the settings above.

In [ ]:
inference_batch = next(iter(validation_loader))
inference_inputs = inference_batch["slices"].to(DEVICE)
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
started = time.perf_counter()
with torch.inference_mode():
    inference_logits = model(inference_inputs)
    inference_probabilities = torch.sigmoid(inference_logits)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
inference_seconds = time.perf_counter() - started
peak_gpu_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if DEVICE.type == "cuda" else 0.0

print(f"Input shape:       {tuple(inference_inputs.shape)}")
print(f"Logit shape:       {tuple(inference_logits.shape)}")
print(f"Probability range: [{inference_probabilities.min().item():.4f}, "
      f"{inference_probabilities.max().item():.4f}]")
print(f"Batch runtime:     {inference_seconds:.3f} s")
print(f"Peak GPU memory:   {peak_gpu_mb:.1f} MB")

## 10. Post-process Model Predictions

Probabilities are thresholded, small 3D components are removed, and at most the two largest connected components are retained for the bilateral target.

In [ ]:
from src.postprocessing.connected_components import keep_largest_k_components


def postprocess_prediction(probabilities: np.ndarray) -> tuple[np.ndarray, int]:
    binary = (probabilities >= PREDICTION_THRESHOLD).astype(np.uint8)
    pruned, removed_count = remove_small_components(binary, MIN_COMPONENT_VOXELS)
    return keep_largest_k_components(pruned, k=2).astype(np.uint8), removed_count


raw_batch_predictions = (inference_probabilities.cpu().numpy() >= PREDICTION_THRESHOLD).astype(np.uint8)
print(f"Raw predicted voxels in batch: {int(raw_batch_predictions.sum())}")

## 11. Visualize Prediction Overlays

Green denotes true positives, red false positives, and blue false negatives. The probability panel helps diagnose overconfident background or underconfident gland predictions.

In [ ]:
def error_overlay(prediction: np.ndarray, target: np.ndarray) -> np.ndarray:
    prediction, target = prediction.astype(bool), target.astype(bool)
    rgba = np.zeros((*prediction.shape, 4), dtype=np.float32)
    rgba[prediction & target] = (0.1, 0.9, 0.2, 0.65)
    rgba[prediction & ~target] = (1.0, 0.1, 0.1, 0.65)
    rgba[~prediction & target] = (0.1, 0.4, 1.0, 0.65)
    return rgba


sample_count = min(6, len(validation_dataset))
positive_sample_indices = [
    index for index, descriptor in enumerate(validation_dataset.samples) if descriptor[2]
][:sample_count]
fig, axes = plt.subplots(len(positive_sample_indices), 4, figsize=(13, 3 * len(positive_sample_indices)), squeeze=False)
model.eval()
for row, sample_index in enumerate(positive_sample_indices):
    sample = validation_dataset[sample_index]
    inputs = sample["slices"].unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        probability = torch.sigmoid(model(inputs))[0, 0].cpu().numpy()
    prediction = probability >= PREDICTION_THRESHOLD
    target = sample["mask"][0].numpy().astype(bool)
    center_image = sample["slices"][SLICE_WINDOW // 2].numpy()

    axes[row, 0].imshow(center_image, cmap="gray")
    axes[row, 0].set_title(f"{sample['case_id']} z={sample['center_index']}")
    axes[row, 1].imshow(target, cmap="gray")
    axes[row, 1].set_title("Ground truth")
    probability_plot = axes[row, 2].imshow(probability, cmap="magma", vmin=0, vmax=1)
    axes[row, 2].set_title("Probability")
    axes[row, 3].imshow(center_image, cmap="gray")
    axes[row, 3].imshow(error_overlay(prediction, target))
    axes[row, 3].set_title("TP green / FP red / FN blue")

for axis in axes.ravel():
    axis.axis("off")
fig.colorbar(probability_plot, ax=axes[:, 2], fraction=0.02, pad=0.02)
plt.show()

## 12. Evaluate Adrenal Gland Segmentation

Full-volume predictions are resized back to the source in-plane dimensions before 3D post-processing. Combined metrics are the valid primary endpoint for this one-channel model. The per-gland rows use the same combined prediction against each gland and are diagnostic only; train a two-channel model for valid left/right predictions.

In [ ]:
def predict_volume(model, case: dict, batch_size=BATCH_SIZE) -> dict:
    image = preprocess_image_volume(case["image"])
    original_height, original_width = image.shape[1:]
    probability_batches = []
    started = time.perf_counter()
    model.eval()

    for start in range(0, image.shape[0], batch_size):
        centers = range(start, min(start + batch_size, image.shape[0]))
        inputs = torch.stack([
            resize_window(make_slice_window(image, center)) for center in centers
        ]).to(DEVICE)
        with torch.inference_mode():
            probabilities = torch.sigmoid(model(inputs))
            probabilities = F.interpolate(
                probabilities,
                size=(original_height, original_width),
                mode="bilinear",
                align_corners=False,
            )
        probability_batches.append(probabilities[:, 0].cpu())

    probability_volume = torch.cat(probability_batches).numpy()
    prediction, removed_count = postprocess_prediction(probability_volume)
    return {
        "probability": probability_volume,
        "prediction": prediction,
        "seconds": time.perf_counter() - started,
        "components_removed": removed_count,
    }


def binary_metrics(prediction: np.ndarray, target: np.ndarray, spacing_zyx) -> dict:
    prediction, target = prediction.astype(bool), target.astype(bool)
    true_positive = int(np.logical_and(prediction, target).sum())
    false_positive = int(np.logical_and(prediction, ~target).sum())
    false_negative = int(np.logical_and(~prediction, target).sum())
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else float("nan")
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else float("nan")
    voxel_volume = float(np.prod(spacing_zyx))
    return {
        "dice": dice_score(prediction, target),
        "iou": iou_score(prediction, target),
        "precision": precision,
        "recall": recall,
        "hd95_mm": hausdorff_distance_95(prediction, target, spacing_zyx),
        "surface_dice_2mm": normalized_surface_dice(prediction, target, 2.0, spacing_zyx),
        "predicted_volume_ml": prediction.sum() * voxel_volume / 1000.0,
        "reference_volume_ml": target.sum() * voxel_volume / 1000.0,
        "volume_error_ml": volume_error(prediction, target, voxel_volume) / 1000.0,
    }


single_validation_case = load_amos_case(selected_validation_records[0])
if RUN_FULL_VOLUME_INFERENCE:
    single_result = predict_volume(model, single_validation_case)
    single_right, single_left, single_combined = extract_adrenal_masks(single_validation_case["label"])
    single_metrics = []
    for gland_name, target, metric_scope in [
        ("combined", single_combined, "primary"),
        ("right", single_right, "diagnostic_combined_prediction"),
        ("left", single_left, "diagnostic_combined_prediction"),
    ]:
        row = binary_metrics(single_result["prediction"], target, single_validation_case["spacing_zyx"])
        row.update({"case_id": single_validation_case["case_id"], "gland": gland_name, "scope": metric_scope})
        single_metrics.append(row)
    single_metrics_df = pd.DataFrame(single_metrics)
    display(single_metrics_df.round(4))
    print(f"Runtime: {single_result['seconds']:.2f}s; components removed: {single_result['components_removed']}")

    informative_slice = int(np.flatnonzero(np.any(single_combined, axis=(1, 2)))[len(np.flatnonzero(np.any(single_combined, axis=(1, 2)))) // 2])
    image_slice = window_ct(single_validation_case["image"][informative_slice])
    prediction_slice = single_result["prediction"][informative_slice]
    target_slice = single_combined[informative_slice]
    fig, axes = plt.subplots(1, 4, figsize=(14, 4), constrained_layout=True)
    axes[0].imshow(image_slice, cmap="gray"); axes[0].set_title("CT")
    axes[1].imshow(target_slice, cmap="gray"); axes[1].set_title("Ground truth")
    axes[2].imshow(prediction_slice, cmap="gray"); axes[2].set_title("Prediction")
    axes[3].imshow(image_slice, cmap="gray"); axes[3].imshow(error_overlay(prediction_slice, target_slice)); axes[3].set_title("Error overlay")
    for axis in axes:
        axis.axis("off")
    plt.show()

## 13. Run Inference on Multiple Cases

Increase `NUM_EVALUATION_CASES` after the smoke test. Errors are captured per case so one malformed scan does not discard completed results.

In [ ]:
NUM_EVALUATION_CASES = 3
metric_rows, case_errors, prediction_cache = [], [], {}

if RUN_FULL_VOLUME_INFERENCE:
    for record in validation_records[:NUM_EVALUATION_CASES]:
        try:
            case = load_amos_case(record)
            result = predict_volume(model, case)
            right, left, combined = extract_adrenal_masks(case["label"])
            prediction_cache[case["case_id"]] = {"case": case, "result": result}
            for gland_name, target, metric_scope in [
                ("combined", combined, "primary"),
                ("right", right, "diagnostic_combined_prediction"),
                ("left", left, "diagnostic_combined_prediction"),
            ]:
                row = binary_metrics(result["prediction"], target, case["spacing_zyx"])
                row.update({
                    "case_id": case["case_id"],
                    "gland": gland_name,
                    "scope": metric_scope,
                    "runtime_seconds": result["seconds"],
                    "components_removed": result["components_removed"],
                })
                metric_rows.append(row)
            print(f"Completed {case['case_id']} in {result['seconds']:.1f}s")
        except Exception as exc:
            case_errors.append({"case_id": record["case_id"], "error": repr(exc)})
            print(f"Failed {record['case_id']}: {exc}")
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

metrics_df = pd.DataFrame(metric_rows)
errors_df = pd.DataFrame(case_errors)
display(metrics_df.round(4))
if not errors_df.empty:
    display(errors_df)

## 14. Summarize and Plot Experiment Metrics

The aggregate table reports mean, standard deviation, median, minimum, and maximum. Primary combined-gland rows are plotted separately from diagnostic per-gland rows.

In [ ]:
if not metrics_df.empty:
    numeric_columns = [
        "dice", "iou", "precision", "recall", "hd95_mm", "surface_dice_2mm",
        "predicted_volume_ml", "reference_volume_ml", "volume_error_ml", "runtime_seconds",
    ]
    summary_df = metrics_df.groupby("gland")[numeric_columns].agg(["mean", "std", "median", "min", "max"])
    display(summary_df.round(4))

    primary_df = metrics_df[metrics_df["gland"] == "combined"].sort_values("case_id")
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    axes[0, 0].boxplot(primary_df["dice"].dropna(), labels=["Combined"])
    axes[0, 0].set(title="Volume Dice distribution", ylabel="Dice", ylim=(0, 1))

    diagnostic = metrics_df[metrics_df["gland"].isin(["right", "left"])]
    for gland, group in diagnostic.groupby("gland"):
        axes[0, 1].plot(group["case_id"], group["dice"], marker="o", label=gland)
    axes[0, 1].set(title="Diagnostic per-reference-gland Dice", xlabel="Case", ylabel="Dice", ylim=(0, 1))
    axes[0, 1].tick_params(axis="x", rotation=45)
    axes[0, 1].legend()

    axes[1, 0].bar(primary_df["case_id"], primary_df["runtime_seconds"])
    axes[1, 0].set(title="Full-volume runtime", xlabel="Case", ylabel="Seconds")
    axes[1, 0].tick_params(axis="x", rotation=45)

    axes[1, 1].scatter(primary_df["reference_volume_ml"], primary_df["predicted_volume_ml"], s=55)
    volume_limit = max(primary_df[["reference_volume_ml", "predicted_volume_ml"]].max().max(), 1.0)
    axes[1, 1].plot([0, volume_limit], [0, volume_limit], linestyle="--", color="gray")
    axes[1, 1].set(title="Reference vs predicted volume", xlabel="Reference (mL)", ylabel="Predicted (mL)")

    for axis in axes.ravel():
        axis.grid(alpha=0.2)
    plt.show()
else:
    print("No metrics to summarize. Set RUN_FULL_VOLUME_INFERENCE=True and run the evaluation cells.")

## 15. Save Predictions and Results

Nothing is written unless `SAVE_OUTPUTS=True`. Saved NIfTI masks reuse each source image’s affine and header. The experiment configuration records the settings needed to reproduce the run.

### Recommended next runs

- Increase `TRAIN_PATIENTS`, `VAL_PATIENTS`, `EPOCHS`, and `IMAGE_SIZE` gradually.
- Compare `resnet18` with the configured `inceptionv4` encoder.
- Keep the official patient-level training/validation separation.
- Enable and validate physical-spacing resampling for final experiments.
- Train with two output channels for valid left-versus-right gland metrics.
- Treat saved checkpoints and predictions as research outputs, not clinically validated results.

In [ ]:
experiment_config = {
    "seed": SEED,
    "device": str(DEVICE),
    "right_adrenal_label": RIGHT_ADRENAL_LABEL,
    "left_adrenal_label": LEFT_ADRENAL_LABEL,
    "slice_window": SLICE_WINDOW,
    "image_size": IMAGE_SIZE,
    "hu_window": HU_WINDOW,
    "train_patients": TRAIN_PATIENTS,
    "validation_patients": VAL_PATIENTS,
    "epochs": EPOCHS,
    "learning_rates": LEARNING_RATES[:NUM_TEST_RUNS],
    "encoder": ENCODER,
    "encoder_weights": ENCODER_WEIGHTS,
    "prediction_threshold": PREDICTION_THRESHOLD,
    "minimum_component_voxels": MIN_COMPONENT_VOXELS,
    "selected_run": best_run,
}

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prediction_dir = OUTPUT_DIR / "predictions"
    prediction_dir.mkdir(exist_ok=True)

    for case_id, cached in prediction_cache.items():
        case, result = cached["case"], cached["result"]
        prediction_xyz = np.transpose(result["prediction"].astype(np.uint8), (1, 2, 0))
        prediction_nifti = nib.Nifti1Image(
            prediction_xyz,
            affine=case["image_nifti"].affine,
            header=case["image_nifti"].header.copy(),
        )
        prediction_nifti.set_data_dtype(np.uint8)
        nib.save(prediction_nifti, prediction_dir / f"{case_id}_adrenal_prediction.nii.gz")

    if not metrics_df.empty:
        metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
        metrics_df.to_json(OUTPUT_DIR / "metrics.json", orient="records", indent=2)
    if not history_df.empty:
        history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

    with (OUTPUT_DIR / "experiment_config.json").open("w", encoding="utf-8") as file:
        json.dump(experiment_config, file, indent=2)

    torch.save(
        {"model_state_dict": model.state_dict(), "config": experiment_config},
        OUTPUT_DIR / "best_model.pt",
    )
    print(f"Saved predictions, metrics, config, and checkpoint to {OUTPUT_DIR}")
else:
    print("SAVE_OUTPUTS=False; no files were written.")